In [2]:
import pandas as pd 

In [3]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01-12-2009 07:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01-12-2009 07:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01-12-2009 07:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01-12-2009 07:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01-12-2009 07:45,1.25,13085.0,United Kingdom


In [4]:
df.shape
df.isnull().sum()

Invoice             0
StockCode           0
Description      4372
Quantity            0
InvoiceDate         0
Price               0
Customer ID    236682
Country             0
dtype: int64

In [5]:
df = df.dropna(subset=["Customer ID"])

In [6]:
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

In [7]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], dayfirst=True)

In [8]:
df = df.drop_duplicates()

In [9]:
df["TotalPrice"] = df["Quantity"] * df["Price"]

In [10]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 767369 entries, 0 to 1048574
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      767369 non-null  object        
 1   StockCode    767369 non-null  object        
 2   Description  767369 non-null  object        
 3   Quantity     767369 non-null  int64         
 4   InvoiceDate  767369 non-null  datetime64[ns]
 5   Price        767369 non-null  float64       
 6   Customer ID  767369 non-null  float64       
 7   Country      767369 non-null  object        
 8   TotalPrice   767369 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 58.5+ MB


,Quantity,InvoiceDate,Price,Customer ID,TotalPrice
count,767369.000000,767369,767369.000000,767369.000000,767369.000000
mean,13.399990,2010-12-28 18:22:29.404392960,3.226796,15320.342234,22.097454
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,2.000000,2010-06-30 13:17:00,1.250000,13971.000000,4.950000
50%,6.000000,2010-11-29 14:21:00,1.950000,15249.000000,12.500000
75%,12.000000,2011-07-21 15:23:00,3.750000,16791.000000,19.800000
max,74215.000000,2011-12-04 13:15:00,10953.500000,18287.000000,77183.600000
std,114.173661,NaN,29.848227,1695.243586,124.271552


**Visualizing Outliers** (to avoid extreme values) : <br>
Lower bound (1%) <br>
Upper bound (99%)

In [11]:
df["Quantity"].quantile([0.01, 0.99])
df["Price"].quantile([0.01, 0.99])

0.01     0.29
0.99    14.95
Name: Price, dtype: float64

**Filtering Outliers** (using percentile-based filtering ): <br>
Reduces skewness and improves model performance.

In [12]:
#for Quantity
q_low = df["Quantity"].quantile(0.01)
q_high = df["Quantity"].quantile(0.99)

df = df[(df["Quantity"] >= q_low) & (df["Quantity"] <= q_high)]

In [13]:
#for price 
p_low = df["Price"].quantile(0.01)
p_high = df["Price"].quantile(0.99)

df = df[(df["Price"] >= p_low) & (df["Price"] <= p_high)]

In [14]:
df["TotalPrice"] = df["Quantity"] * df["Price"]

In [15]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID,TotalPrice
count,748222.000000,748222,748222.000000,748222.000000,748222.000000
mean,10.024192,2010-12-28 14:14:14.093650944,2.818129,15326.613792,18.457991
min,1.000000,2009-12-01 07:45:00,0.290000,12346.000000,0.290000
25%,2.000000,2010-06-30 13:03:00,1.250000,13979.000000,4.950000
50%,5.000000,2010-11-29 13:42:00,1.950000,15260.000000,11.900000
75%,12.000000,2011-07-21 13:37:00,3.750000,16794.000000,19.500000
max,144.000000,2011-12-04 13:15:00,14.950000,18287.000000,1576.800000
std,15.809903,NaN,2.574032,1691.867907,32.260471


FEATURE ENGINEERING <br>
Creating time-based features for analysis.

In [16]:
df["Month"] = df["InvoiceDate"].dt.month
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek

In [18]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)